# Preparación de datos: química natural y sintética

Se muestran las observaciones disponibles para la próxima calibración: **LAB004–009 de mosto natural** y **MS007–016 de mosto sintético**. LAB010–012 quedan fuera de esta selección, conforme al alcance acordado. Los parámetros no se ajustan en este notebook.

Por cada variable se presenta primero la matriz de paneles naturales (2 × 3) y después la sintética (2 × 5), en este orden: **biomasa viable, YAN, glucosa, fructosa, glucosa + fructosa, densidad y etanol**. Cada marcador es una observación disponible; los segmentos unen puntos en orden temporal y no son simulaciones ni muestras interpoladas. `n` indica los puntos realmente dibujados. Las escalas verticales de una misma variable son comunes a ambas matrices; el tiempo comparte escala dentro de cada matriz y conserva su referencia histórica.

La biomasa viable es la usada por el estado X (30 pg/célula; g/L = kg/m³); YAN se muestra en mg N/L, azúcares y etanol en g/L, y densidad en las unidades originales de la tabla. G+F se calcula únicamente cuando ambas mediciones existen en la misma fila, sin completar faltantes con densidad. La densidad se incluye como control experimental de la fermentación, no como un estado adicional de las ecuaciones. Se omiten biomasa muerta y glicerol de esta visualización por el foco solicitado; esto no modifica los estados del modelo vigente ni sus parámetros.

Se consume `secondary_metabolite_long.csv`, sin depender de salidas de otros notebooks. Los datos faltantes no se rellenan ni se convierten en cero. Las figuras y tablas permanecen dentro del EXECUTED: no se exportan CSV ni imágenes al proyecto.

## Referencias de nutrición

En **naturales**, la línea vertical identifica el segundo pulso según `Fernanda Folch.ics`, convertido al reloj histórico de cada LAB. Es una referencia de calendario, no una confirmación de ejecución. No se reemplaza por la fila de pulso del modelo, que tiene horarios distintos.

En **sintéticos**, la línea marca el pulso de N registrado en el CSV normalizado (`N_pulse_mg_l > 0`), es decir, la entrada registrada para el modelo. No se presupone que ambas fuentes tengan el mismo nivel de confirmación. Las tablas inferiores identifican horarios y dosis disponibles.

### Exclusiones de biomasa confirmadas

Se excluyen de la tabla de biomasa preparada y de sus figuras seis mediciones señaladas por el usuario: LAB004-10, LAB005-10 y LAB006-09 (115 h), y las últimas de LAB007/008 (L-13) y LAB009 (L-12), a las 160 h. Son exclusiones por revisión de calidad, no una detección estadística automática. La identificación usa LAB + ID + tiempo para evitar confundir números de muestra.

Los datos Oculyze de esas seis muestras se eliminaron del Excel fuente (concentraciones viable y total, y gemación) y sus concentraciones/biomasas derivadas se vaciaron en ambos CSV normalizados. Se conservan las filas y todas las demás mediciones, incluido el peso seco independiente. Las correcciones manuales de la muestra 4 de LAB004, LAB005 y LAB006 permanecen aplicadas. La tabla inferior documenta los identificadores eliminados y sus motivos, sin conservar sus valores antiguos. `biomass` contiene las 70 mediciones disponibles de la fuente ya depurada; no aplica una máscara adicional ni interpola puntos. Esta depuración no reejecuta las calibraciones ni otros notebooks.


In [ ]:
from pathlib import Path
import hashlib
import pandas as pd
import matplotlib.pyplot as plt
from IPython import get_ipython
from IPython.display import display

get_ipython().run_line_magic('matplotlib', 'inline')
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / 'fermentation_model/shared').is_dir())
DATA_PATH = ROOT / 'fermentation_model/shared/results/secondary_metabolite_data_review/secondary_metabolite_long.csv'
input_hash = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
LABS = tuple(f'LAB{i:03d}' for i in range(4, 10))
BIOMASS_COLUMN = 'X_viable_kg_m3'

raw = pd.read_csv(DATA_PATH)
biomass = raw.loc[raw.medium.eq('natural') & raw.batch.isin(LABS),
                  ['batch', 'sample_id', 'time_h', BIOMASS_COLUMN]].copy()
for column in ['time_h', BIOMASS_COLUMN]:
    biomass[column] = pd.to_numeric(biomass[column], errors='coerce')
# Trazabilidad de la depuración ya aplicada a las fuentes; no es un filtro del notebook.
exclusion_specs = (
    ('LAB004', 'ING26-LAB004-10', 115.0, 'Caída aislada con recuperación posterior'),
    ('LAB005', 'ING26-LAB005-10', 115.0, 'Caída aislada con recuperación posterior'),
    ('LAB006', 'ING26-LAB006-09', 115.0, 'Caída aislada con recuperación posterior'),
    ('LAB007', 'ING26-LAB007-L-13', 160.0, 'Última muestra: caída abrupta; exclusión confirmada'),
    ('LAB008', 'ING26-LAB008-L-13', 160.0, 'Última muestra: caída abrupta; exclusión confirmada'),
    ('LAB009', 'ING26-LAB009-L-12', 160.0, 'Última muestra: caída abrupta; exclusión confirmada'),
)
biomass_fields = ['X_viable_mcells_ml', 'X_total_mcells_ml', 'X_dead_direct_mcells_ml',
                 'X_dead_mcells_ml', 'X_viable_kg_m3', 'X_total_kg_m3', 'X_dead_kg_m3']
removal_rows = []
for lab, sample_id, time_h, reason in exclusion_specs:
    match = (raw.medium.eq('natural') & raw.batch.eq(lab) & raw.sample_id.eq(sample_id)
             & raw.time_h.sub(time_h).abs().lt(1e-6))
    assert match.sum() == 1, (lab, sample_id, time_h)
    assert raw.loc[match, biomass_fields].isna().all().all()
    removal_rows.append({'batch':lab, 'sample_id':sample_id, 'time_h':time_h,
                         'estado':'Oculyze eliminado de la fuente', 'motivo':reason})
removed_oculyze = pd.DataFrame(removal_rows)
biomass = biomass.dropna(subset=['time_h', BIOMASS_COLUMN]).sort_values(['batch', 'time_h'])
assert len(removed_oculyze) == 6 and len(biomass) == 70
# Las tres correcciones manuales siguen presentes.
for lab, expected in [('LAB004', 1.044), ('LAB005', 1.5576), ('LAB006', 0.9195)]:
    point = biomass.loc[biomass.batch.eq(lab) & biomass.sample_id.eq(f'ING26-{lab}-04')]
    assert len(point) == 1 and abs(point.iloc[0][BIOMASS_COLUMN]-expected) < 1e-12
print('Mediciones Oculyze eliminadas de la fuente (otras variables conservadas):')
display(removed_oculyze)
assert tuple(sorted(biomass.batch.unique())) == LABS
assert not biomass.duplicated(['batch', 'time_h']).any()
print('Fuente:', DATA_PATH.relative_to(ROOT))
print('Muestras por LAB:', biomass.groupby('batch').size().to_dict())
# Referencia temporal: el mismo t0 archivado para estas series históricas.
import re
CALENDAR_PATH = ROOT / 'fermentation_model/data/Laboratorio 2026/raw_data/Fernanda Folch.ics'
CLOCK_PATH = ROOT / 'fermentation_model/laboratory_2026/results/estimability_historical_natural/process_temperature_30min.csv'
nutrition_input_hashes = {p:hashlib.sha256(p.read_bytes()).hexdigest() for p in (CALENDAR_PATH,CLOCK_PATH)}
clocks = pd.read_csv(CLOCK_PATH,usecols=['batch','t0']).drop_duplicates()
clocks = clocks.loc[clocks.batch.isin(LABS)].set_index('batch')
assert clocks.index.is_unique and set(clocks.index)==set(LABS)
calendar_text = re.sub(r'\r?\n[ \t]','',CALENDAR_PATH.read_text(encoding='utf-8-sig'))
nutrition_rows = []
for event in calendar_text.split('BEGIN:VEVENT')[1:]:
    event = event.split('END:VEVENT')[0]
    fields = dict(line.split(':',1) for line in event.splitlines() if ':' in line)
    if 'Pulso nutricional 2' not in fields.get('SUMMARY',''):
        continue
    match = re.search(r'LAB00[4-9]',event)
    if match is None:
        continue
    lab = match.group()
    event_utc = pd.to_datetime(fields['DTSTART'],format='%Y%m%dT%H%M%SZ',utc=True)
    t0_local = pd.Timestamp(clocks.loc[lab,'t0']).tz_localize('America/Santiago')
    nutrition_rows.append({'LAB':lab,'hora_calendario_local':event_utc.tz_convert('America/Santiago').isoformat(),
        't_nutricion_h':(event_utc-t0_local).total_seconds()/3600,
        'fuente':'Calendario operativo · pulso 2'})
nutrition = pd.DataFrame(nutrition_rows).sort_values('LAB').reset_index(drop=True)
assert tuple(nutrition.LAB)==LABS
nutrition_times = nutrition.set_index('LAB').t_nutricion_h.to_dict()
display(nutrition)

# Selección de ambas matrices y variables relevantes: solo observaciones disponibles.
SYNTHETIC_LABS = tuple(f'MS{i:03d}' for i in range(7, 17))
MATRICES = {'natural': LABS, 'synthetic': SYNTHETIC_LABS}
selected = raw.loc[(raw.medium.eq('natural') & raw.batch.isin(LABS))
                   | (raw.medium.eq('synthetic') & raw.batch.isin(SYNTHETIC_LABS))].copy()
VARIABLES = (
    ('X_viable_kg_m3', 'Biomasa viable', 'Biomasa viable [g/L]'),
    ('YAN_mg_l', 'Nitrógeno asimilable (YAN)', 'YAN [mg N/L]'),
    ('G_g_l', 'Glucosa', 'Glucosa [g/L]'),
    ('F_g_l', 'Fructosa', 'Fructosa [g/L]'),
    ('GF_g_l', 'Azúcar total: glucosa + fructosa', 'G+F [g/L]'),
    ('density', 'Densidad', 'Densidad [kg/m³; equivalente a g/L]'),
    ('E_g_l', 'Etanol', 'Etanol [g/L]'),
)
for column in ['time_h', 'N_pulse_mg_l', 'S_GF_g_l'] + [v[0] for v in VARIABLES if v[0] != 'GF_g_l']:
    selected[column] = pd.to_numeric(selected[column], errors='coerce')
selected['GF_g_l'] = selected['G_g_l'] + selected['F_g_l']
assert selected['GF_g_l'].isna().equals(selected['S_GF_g_l'].isna())
assert selected['GF_g_l'].sub(selected['S_GF_g_l']).dropna().abs().lt(1e-9).all()
assert selected.time_h.notna().all()
assert not selected.duplicated(['medium', 'batch', 'time_h']).any()
assert len(selected) == 181  # 85 filas naturales, 96 sintéticas; incluye filas de pulsos.
for medium, labs in MATRICES.items():
    assert set(selected.loc[selected.medium.eq(medium), 'batch']) == set(labs)
synthetic_nutrition = selected.loc[selected.medium.eq('synthetic') & selected.N_pulse_mg_l.gt(0),
                                   ['batch', 'sample_id', 'time_h', 'N_pulse_mg_l']].copy()
assert tuple(sorted(synthetic_nutrition.batch.unique())) == SYNTHETIC_LABS
print('Pulsos de N en sintéticos: tabla normalizada (dosis en mg N/L).')
display(synthetic_nutrition.reset_index(drop=True))
coverage = selected.groupby(['medium', 'batch'])[[v[0] for v in VARIABLES]].count()
coverage.columns = [v[1] for v in VARIABLES]
print('Cantidad de mediciones disponibles por experimento y variable:')
display(coverage)
plot_audit = []


In [ ]:
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator

plt.style.use('seaborn-v0_8-whitegrid')

def density_crossing_for_lab(lab, target=1040.):
    samples = selected.loc[selected.medium.eq('natural') & selected.batch.eq(lab),
                           ['sample_id', 'time_h', 'density']].dropna().sort_values('time_h')
    candidates = []
    for i in range(len(samples)):
        left = samples.iloc[i]
        if abs(left.density-target) <= 1e-9:
            candidates.append((float(left.time_h), left, left))
        if i+1 < len(samples):
            right = samples.iloc[i+1]
            if (left.density-target)*(right.density-target) < 0:
                t = left.time_h+(target-left.density)*(right.time_h-left.time_h)/(right.density-left.density)
                candidates.append((float(t), left, right))
    assert candidates, f'Sin cruce de densidad {target}: {lab}'
    t, left, right = min(candidates, key=lambda value: abs(value[0]-nutrition_times[lab]))
    assert left.time_h <= t <= right.time_h
    return {'LAB':lab, 'calendario_h':nutrition_times[lab], 'cruce_1040_h':t,
            'cruce_menos_calendario_h':t-nutrition_times[lab],
            'muestra_izquierda':left.sample_id, 'muestra_derecha':right.sample_id,
            'intervalo_desde_h':float(left.time_h), 'intervalo_hasta_h':float(right.time_h),
            'ancho_intervalo_h':float(right.time_h-left.time_h)}

density_crossings = pd.DataFrame([density_crossing_for_lab(lab) for lab in LABS])
density_times = density_crossings.set_index('LAB').cruce_1040_h.to_dict()
print('Calendario frente a cruce de densidad 1040: referencias de tiempo, no confirmación de dosificación.')
display(density_crossings.round(3))

def plot_matrix(column, label, ylabel, medium):
    labs = MATRICES[medium]
    group = selected.loc[selected.medium.eq(medium)]
    is_natural = medium == 'natural'
    compare_timing = is_natural and column == 'YAN_mg_l'
    figsize = (14, 8) if is_natural else (19, 8)
    fig, axes = plt.subplots(2, 3 if is_natural else 5, figsize=figsize,
                             sharex=True, sharey=True, layout='constrained')
    values = selected[column].dropna().to_numpy(float)
    assert len(values) and np.isfinite(values).all()
    lower, upper = float(values.min()), float(values.max())
    span = max(upper-lower, abs(upper)*0.05, 1e-6)
    # Margen mínimo bajo cero cuando corresponde: permite ver líneas en cero.
    limits = ((lower-0.05*span, upper+0.08*span) if column == 'density'
              else (min(0., lower)-0.02*span, upper+0.08*span))
    xmax = float(group.time_h.max())
    counts = {}
    for ax, lab, color in zip(axes.flat, labs, plt.get_cmap('tab10').colors):
        samples = group.loc[group.batch.eq(lab) & group[column].notna()].sort_values('time_h')
        assert samples.time_h.is_unique and len(samples) > 0
        line, = ax.plot(samples.time_h, samples[column], color=color,
                        marker='o', markersize=4.5, linewidth=1.6, zorder=3)
        assert np.array_equal(line.get_xdata(), samples.time_h.to_numpy())
        assert np.array_equal(line.get_ydata(), samples[column].to_numpy())
        counts[lab] = len(samples)
        times = ([nutrition_times[lab]] if is_natural else
                 synthetic_nutrition.loc[synthetic_nutrition.batch.eq(lab), 'time_h'].tolist())
        for t_nut in times:
            ax.axvline(t_nut, color='#555555', linestyle='--', linewidth=1.1, zorder=2)
            ax.text(t_nut+0.015*xmax, 0.96, (f'Cal.: {t_nut:g} h' if compare_timing else f'N: {t_nut:g} h'),
                    transform=ax.get_xaxis_transform(), va='top', ha='left', fontsize=7,
                    bbox=dict(facecolor='white', edgecolor='none', alpha=0.8, pad=1.5))
        if compare_timing:
            crossing = density_times[lab]
            crossing_line = ax.axvline(crossing, color='#C000A0', linestyle=':', linewidth=1.7, zorder=2)
            assert np.allclose(crossing_line.get_xdata(), [crossing, crossing])
            ax.text(crossing+0.015*xmax, 0.86, f'1040: {crossing:.2f} h',
                    transform=ax.get_xaxis_transform(), va='top', ha='left', fontsize=7,
                    color='#A00080',
                    bbox=dict(facecolor='white', edgecolor='none', alpha=0.85, pad=1.5))
        ax.set_title(f'{lab} · n={len(samples)}', fontsize=11)
        ax.set_xlim(-0.02*xmax, 1.02*xmax)
        ax.set_ylim(*limits)
        ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=6))
        ax.tick_params(labelbottom=True, labelleft=True, labelsize=9)
        ax.grid(alpha=0.3)
    matrix_label = 'Mosto natural · LAB004–009' if is_natural else 'Mosto sintético · MS007–016'
    pulse_label = 'Nutrición 2 · calendario operativo' if is_natural else 'Pulso N · tabla normalizada'
    fig.suptitle(f'{label} · {matrix_label}', fontsize=14, y=0.995)
    fig.supxlabel('Tiempo histórico [h]')
    fig.supylabel(ylabel)
    handles = [Line2D([], [], color='#333333', marker='o', lw=1.6, label='Muestras disponibles'),
               Line2D([], [], color='#555555', ls='--', lw=1.1, label=pulse_label)]
    if compare_timing:
        handles.append(Line2D([], [], color='#C000A0', ls=':', lw=1.7, label='Cruce de densidad 1040 · estimado'))
    fig.get_layout_engine().set(rect=(0, 0, 1, 0.90))
    fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 0.965),
               ncol=3 if compare_timing else 2, frameon=False, fontsize=9)
    plot_audit.append({'variable':column, 'medium':medium, 'counts':counts, 'ylim':limits,
                       'n_panels':len(axes.flat), 'compares_density_1040':compare_timing})
    plt.show()
    plt.close(fig)


## 1. Biomasa viable

Las seis mediciones Oculyze descartadas ya están ausentes de la fuente. Se conservan las muestras corregidas de LAB004–006.

Primero mosto natural; después mosto sintético.

In [ ]:
plot_matrix('X_viable_kg_m3', 'Biomasa viable', 'Biomasa viable [g/L]', 'natural')

In [ ]:
plot_matrix('X_viable_kg_m3', 'Biomasa viable', 'Biomasa viable [g/L]', 'synthetic')

## 2. Nitrógeno asimilable (YAN)

Se muestran concentraciones medidas; las dosis de los pulsos no se dibujan como mediciones de YAN.

Primero mosto natural; después mosto sintético.

En la matriz **natural de YAN** se comparan dos referencias: calendario (gris discontinuo) y cruce de densidad 1040 (magenta punteado). El cruce se obtiene por interpolación lineal entre muestras de densidad, o directamente si existe una lectura de 1040, como en LAB006. Las anotaciones tienen alturas distintas para distinguir referencias próximas. La tabla de comparación conserva el intervalo de mediciones usado: los minutos/decimales no son una precisión experimental ni confirman la hora de nutrición. Las demás figuras conservan sus referencias anteriores.


In [ ]:
plot_matrix('YAN_mg_l', 'Nitrógeno asimilable (YAN)', 'YAN [mg N/L]', 'natural')

In [ ]:
plot_matrix('YAN_mg_l', 'Nitrógeno asimilable (YAN)', 'YAN [mg N/L]', 'synthetic')

## 3. Glucosa



Primero mosto natural; después mosto sintético.

In [ ]:
plot_matrix('G_g_l', 'Glucosa', 'Glucosa [g/L]', 'natural')

In [ ]:
plot_matrix('G_g_l', 'Glucosa', 'Glucosa [g/L]', 'synthetic')

## 4. Fructosa



Primero mosto natural; después mosto sintético.

In [ ]:
plot_matrix('F_g_l', 'Fructosa', 'Fructosa [g/L]', 'natural')

In [ ]:
plot_matrix('F_g_l', 'Fructosa', 'Fructosa [g/L]', 'synthetic')

## 5. Azúcar total: glucosa + fructosa

La suma G+F usa únicamente pares medidos en la misma muestra. No se infiere azúcar desde densidad.

Primero mosto natural; después mosto sintético.

In [ ]:
plot_matrix('GF_g_l', 'Azúcar total: glucosa + fructosa', 'G+F [g/L]', 'natural')

In [ ]:
plot_matrix('GF_g_l', 'Azúcar total: glucosa + fructosa', 'G+F [g/L]', 'synthetic')

## 6. Densidad



Primero mosto natural; después mosto sintético.

In [ ]:
plot_matrix('density', 'Densidad', 'Densidad [kg/m³; equivalente a g/L]', 'natural')

In [ ]:
plot_matrix('density', 'Densidad', 'Densidad [kg/m³; equivalente a g/L]', 'synthetic')

## 7. Etanol

Se usa E_g_l, la concentración de etanol normalizada que consume el modelo. Los sintéticos tienen cuatro puntos por experimento; las líneas solo unen esas mediciones.

Primero mosto natural; después mosto sintético.

In [ ]:
plot_matrix('E_g_l', 'Etanol', 'Etanol [g/L]', 'natural')

In [ ]:
plot_matrix('E_g_l', 'Etanol', 'Etanol [g/L]', 'synthetic')

## Verificación y alcance

Esta vista reúne 16 experimentos y siete variables, sin ajustar parámetros ni modificar las fuentes. Los recuentos de cada panel se contrastan con la tabla de cobertura, y se comprueba que las escalas verticales coincidan entre matrices. Los faltantes permanecen ausentes. Las líneas entre muestras no permiten afirmar qué ocurrió dentro de los intervalos sin medición; esto es especialmente relevante para el etanol sintético. Glicerol y biomasa muerta no se muestran por el alcance solicitado, aunque el modelo actual todavía los contiene.

In [ ]:
assert len(plot_audit) == 14
expected_order = [(column, medium) for column, _, _ in VARIABLES for medium in MATRICES]
assert [(row['variable'], row['medium']) for row in plot_audit] == expected_order
for i, (column, _, _) in enumerate(VARIABLES):
    natural, synthetic = plot_audit[2*i:2*i+2]
    assert natural['ylim'] == synthetic['ylim']
    for entry in (natural, synthetic):
        medium = entry['medium']
        assert entry['n_panels'] == len(MATRICES[medium])
        expected = selected.loc[selected.medium.eq(medium)].groupby('batch')[column].count().to_dict()
        assert entry['counts'] == expected
assert sum(plot_audit[0]['counts'].values()) == 70
assert all(n == 4 for n in plot_audit[-1]['counts'].values())
assert hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() == input_hash
assert all(hashlib.sha256(p.read_bytes()).hexdigest()==h for p,h in nutrition_input_hashes.items())
print('Verificado: 16 experimentos, 7 variables, 14 figuras y 112 paneles; fuentes intactas.')

assert [row['variable'] for row in plot_audit if row['compares_density_1040']] == ['YAN_mg_l']
assert all(row['medium']=='natural' for row in plot_audit if row['compares_density_1040'])
assert len(density_crossings)==6 and abs(density_times['LAB006']-74.)<1e-9
